In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import numpy as np

# 3 tane eğitim görüntüsü (gerçekte 3072 piksel, biz 2 piksel kullanalım)
X_train = np.array([
    [10, 20],   # 1. görüntü → kedi
    [15, 25],   # 2. görüntü → kedi
    [80, 90],   # 3. görüntü → köpek
])
y_train = np.array([0, 0, 1])  # 0=kedi, 1=köpek

# 1 tane test görüntüsü — bu ne?
X_test = np.array([[12, 22]])

# Her eğitim görüntüsüne mesafeyi hesapla
for i, x in enumerate(X_train):
    mesafe = np.sqrt(np.sum((X_test[0] - x) ** 2))
    print(f"Eğitim {i} → mesafe: {mesafe:.2f}, etiket: {y_train[i]}")

Eğitim 0 → mesafe: 2.83, etiket: 0
Eğitim 1 → mesafe: 4.24, etiket: 0
Eğitim 2 → mesafe: 96.17, etiket: 1


Test görüntüsü en çok neye benziyor ?
Mesafe küçükse birbirine benziyor, mesafe büyükse birbirine benzemiyor.


In [2]:
# En yakın komşunun indeksini bul
mesafeler = []
for i, x in enumerate(X_train):
    mesafe = np.sqrt(np.sum((X_test[0] - x) ** 2))
    mesafeler.append(mesafe)

mesafeler = np.array(mesafeler)
print("Tüm mesafeler:", mesafeler)

# En küçük mesafenin indeksi
en_yakin = np.argmin(mesafeler)
print("En yakın komşu indeksi:", en_yakin)
print("Tahmin:", "kedi" if y_train[en_yakin] == 0 else "köpek")

Tüm mesafeler: [ 2.82842712  4.24264069 96.16652224]
En yakın komşu indeksi: 0
Tahmin: kedi


In [3]:
k = 3

# En yakın k komşunun indekslerini bul
# argsort → küçükten büyüğe sıralar, ilk k tanesini al
en_yakin_k = np.argsort(mesafeler)[:k]
print("En yakın 3 komşunun indeksleri:", en_yakin_k)

# Bu komşuların etiketleri
komsularin_etiketleri = y_train[en_yakin_k]
print("Komşuların etiketleri:", komsularin_etiketleri)
print("(0=kedi, 1=köpek)")

# Çoğunluk oyu
# bincount → her etiketten kaç tane var sayar
oylar = np.bincount(komsularin_etiketleri)
print("Oylar (kedi / köpek):", oylar)

tahmin = np.argmax(oylar)
print("Tahmin:", "kedi" if tahmin == 0 else "köpek")

En yakın 3 komşunun indeksleri: [0 1 2]
Komşuların etiketleri: [0 0 1]
(0=kedi, 1=köpek)
Oylar (kedi / köpek): [2 1]
Tahmin: kedi


1. Mesafeleri hesapla   →  [ 2.83,  4.24,  96.17 ]
2. En yakın k'yı bul   →  indeksler: [0, 1, 2]
3. Çoğunluk oyu        →  kedi:2, köpek:1  →  kedi!

In [4]:
# Şimdiye kadar yaptığımızı tek fonksiyon haline getirelim
def knn_tahmin(X_train, y_train, X_test, k=1):
    tahminler = []

    for test in X_test:
        # 1. Tüm mesafeleri hesapla
        mesafeler = np.sqrt(np.sum((X_train - test) ** 2, axis=1))

        # 2. En yakın k komşuyu bul
        en_yakin_k = np.argsort(mesafeler)[:k]
        komsular = y_train[en_yakin_k]

        # 3. Çoğunluk oyu
        tahmin = np.argmax(np.bincount(komsular))
        tahminler.append(tahmin)

    return np.array(tahminler)

# Test edelim
tahminler = knn_tahmin(X_train, y_train, X_test, k=3)
print("Tahmin:", "kedi" if tahminler[0] == 0 else "köpek")

Tahmin: kedi


In [7]:
# Two Loop (En Yavaş)
# X_train: 3 görüntü, X_test: 1 görüntü
# Sonuç: (1, 3) boyutunda mesafe matrisi

num_test  = X_test.shape[0]   # 1
num_train = X_train.shape[0]  # 3

dists = np.zeros((num_test, num_train))

for i in range(num_test):      # i = 0 (1 test görüntüsü)
    for j in range(num_train): # j = 0, 1, 2 (3 eğitim görüntüsü)
        dists[i, j] = np.sqrt(np.sum((X_test[i] - X_train[j]) ** 2))

print("İki döngü sonucu:")
print(dists)

İki döngü sonucu:
[[ 2.82842712  4.24264069 96.16652224]]


In [8]:
# One Loop (Broadcasting)
dists_one = np.zeros((num_test, num_train))

for i in range(num_test):  # sadece test üzerinde döngü, train yok!

    # X_test[i] shape: (2,)
    # X_train shape:   (3, 2)
    #
    # X_test[i] - X_train → numpy X_test[i]'yi 3 kez "kopyalar":
    #   [12, 22]        [12, 22]        [12, 22]
    # - [10, 20]      - [15, 25]      - [80, 90]
    # = [ 2,  2]      = [-3, -3]      = [-68, -68]
    #
    # Tek satırda 3 eğitim görüntüsüyle fark hesaplandı!
    # axis=1 → her satırı topla → shape (3,)

    dists_one[i, :] = np.sqrt(np.sum((X_test[i] - X_train) ** 2, axis=1))

print("Tek döngü sonucu:")
print(dists_one)

print("\nİki döngü ile aynı mı?", np.allclose(dists, dists_one))

Tek döngü sonucu:
[[ 2.82842712  4.24264069 96.16652224]]

İki döngü ile aynı mı? True


In [10]:
# No Loop (Matris Çarpımı)
# (a - b)² = a² - 2ab + b²


# ADIM 1: Her test görüntüsünün karelerini topla
# X_test = [[12, 22]]
# 12² + 22² = 144 + 484 = 628
# keepdims=True → shape (1,1) olsun, broadcast için
test_sq = np.sum(X_test ** 2, axis=1, keepdims=True)
print("test_sq:", test_sq)

# ADIM 2: Her eğitim görüntüsünün karelerini topla
# [10²+20², 15²+25², 80²+90²] = [500, 850, 14500]
# shape: (3,)
train_sq = np.sum(X_train ** 2, axis=1)
print("train_sq:", train_sq)

# ADIM 3: Çapraz çarpım X * X_train^T
# shape: (1,2) x (2,3) = (1,3)
cross = X_test @ X_train.T
print("cross:", cross)

# ADIM 4: Hepsini birleştir
# a² - 2ab + b² = (a-b)²
dists_no = np.sqrt(np.maximum(test_sq + train_sq - 2 * cross, 0))
print("\nHiç döngü sonucu:")
print(dists_no)

print("İki döngü ile aynı mı?", np.allclose(dists, dists_no))

test_sq: [[628]]
train_sq: [  500   850 14500]
cross: [[ 560  730 2940]]

Hiç döngü sonucu:
[[ 2.82842712  4.24264069 96.16652224]]
İki döngü ile aynı mı? True


In [11]:
import time

# Büyük veri oluştur (gerçek assignment boyutu)
X_train_buyuk = np.random.randn(5000, 3072)
X_test_buyuk  = np.random.randn(500, 3072)

# İki döngü — çok yavaş olduğu için sadece 10 test ile dene
baslangic = time.time()
dists_temp = np.zeros((10, 5000))
for i in range(10):
    for j in range(5000):
        dists_temp[i,j] = np.sqrt(np.sum((X_test_buyuk[i] - X_train_buyuk[j])**2))
iki_sure = time.time() - baslangic
print(f"İki döngü (10 test): {iki_sure:.2f} saniye")
print(f"500 test için tahminen: {iki_sure*50:.1f} saniye")

# Hiç döngü — tüm 500 test
baslangic = time.time()
test_sq  = np.sum(X_test_buyuk**2, axis=1, keepdims=True)
train_sq = np.sum(X_train_buyuk**2, axis=1)
cross    = X_test_buyuk @ X_train_buyuk.T
dists_hiz = np.sqrt(np.maximum(test_sq + train_sq - 2*cross, 0))
hiz_sure = time.time() - baslangic
print(f"\nHiç döngü (500 test): {hiz_sure:.2f} saniye")

İki döngü (10 test): 0.80 saniye
500 test için tahminen: 40.1 saniye

Hiç döngü (500 test): 0.60 saniye
